# 01 — Data Cleaning
**Online Retail II** (UCI): ~1.07M raw invoice lines, Dec 2009 - Dec 2011.
Every drop below is deliberate, counted, and justified — cleaning IS analysis.

In [1]:
import sys, os
sys.path.append(os.path.abspath(".."))
import pandas as pd
from src.rfm import load_raw, clean_transactions

raw = load_raw()
print("raw shape:", raw.shape)
raw.head()

raw shape: (1067371, 8)


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,CustomerID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom


**Cleaning rules**
1. Invoices starting with `C` are cancellations (credit notes) — not purchases.
2. Non-positive Quantity/Price — returns and manual adjustment postings.
3. Missing `Customer ID` (~20% of lines) — anonymous till transactions cannot be segmented.
4. Non-product stock codes (POST, DOT, M, BANK CHARGES...) — fees would inflate Monetary.

In [2]:
tx = clean_transactions(raw)
print(f"date span: {tx['InvoiceDate'].min().date()} .. {tx['InvoiceDate'].max().date()}")
print(f"customers: {tx['CustomerID'].nunique():,} | invoices: {tx['Invoice'].nunique():,}")
tx.head()

clean_transactions(): start 1,067,371
  - cancelled invoices:       -19,494
  - non-positive qty/price:   -6,207
  - missing CustomerID:       -236,121
  - non-product stock codes:  -2,915
  = kept 802,634 rows (75.2%)
date span: 2009-12-01 .. 2011-12-09
customers: 5,852 | invoices: 36,594


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,CustomerID,Country,Revenue
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085,United Kingdom,83.4
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085,United Kingdom,81.0
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085,United Kingdom,81.0
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085,United Kingdom,100.8
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085,United Kingdom,30.0


In [3]:
# Outlier sanity check: the largest single lines (wholesale spikes are real
# in this dataset — kept, but log-transformed before clustering)
tx.nlargest(5, "Revenue")[["Invoice", "StockCode", "Description", "Quantity", "Price", "Revenue"]]

,Invoice,StockCode,Description,Quantity,Price,Revenue
802166,581483,23843,"PAPER CRAFT , LITTLE BIRDIE",80995,2.08,168469.6
443271,541431,23166,MEDIUM CERAMIC TOP STORAGE JAR,74215,1.04,77183.6
561063,556444,22502,PICNIC BASKET WICKER 60 PIECES,60,649.50,38970.0
337119,530715,84347,ROTATING SILVER ANGELS T-LIGHT HLDR,9360,1.69,15818.4
171893,511465,15044A,PINK PAPER PARASOL,3500,2.55,8925.0
